# Seasonal Vecchia AR(1)+Daily Fit — Apr–Sep

Fits `VecchiaAR1Daily` (from `kernels_vecchia_seasonal.py`) on one year of Apr–Sep GEMS data.

**Conditioning set:**  
- G1 (t=0): k_space spatial neighbors at t  
- G2 (0<t<8): k@t + self@t-1 + k@t-1  
- G3 (t≥8): k@t + self@t-1 + k@t-1 + self@t-8 + k@t-8  

**Computational note (Mac CPU):**  
Full 6-month fit (~1464 time steps × ~3000 obs ≈ 4M obs total) is memory-feasible but slow.  
Use `stride_days=7` for a quick test (~26 days selected); set `stride_days=1` for full fit.

In [1]:
import sys
import time
import math
import pickle
import numpy as np
import pandas as pd
import torch
from pathlib import Path

gems_tco_path = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
sys.path.append(gems_tco_path)

from GEMS_TCO import configuration as config
from GEMS_TCO import orderings
from GEMS_TCO.kernels_vecchia_seasonal import (
    VecchiaAR1Daily,
    load_year_for_seasonal,
    _to_physical,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cpu


## 1. Configuration

In [2]:
# ── Choose year & months ──────────────────────────────────────────────────────
YEAR   = 2024
MONTHS = [7]         # None = Apr–Sep; [7] = July only

# ── Subsampling (set stride_days=1, max_days=None for full fit) ───────────────
STRIDE_DAYS = 1
MAX_DAYS    = None

# ── Spatial domain (same as daily model) ─────────────────────────────────────
LAT_RANGE = [-3, 1]
LON_RANGE = [121, 125]

# ── Vecchia settings ──────────────────────────────────────────────────────────
K_SPACE        = 6      # spatial neighbors (k_s)
MM_COND_NUMBER = 30     # MaxMin NNS: keep top-30 spatial neighbors
SMOOTH         = 0.5    # Matérn ν

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR  = Path(config.mac_data_load_path) / "Apr_to_Sep"
OUT_DIR   = Path("/Users/joonwonlee/Documents/GEMS_TCO-1/outputs/seasonal")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Year={YEAR}  months={MONTHS}  stride={STRIDE_DAYS}  max_days={MAX_DAYS}")
print(f"Data dir: {DATA_DIR}")

Year=2024  months=[7]  stride=1  max_days=None
Data dir: /Users/joonwonlee/Documents/GEMS_DATA/Apr_to_Sep


## 2. Monthly means lookup

In [3]:
mm_df = pd.read_csv(DATA_DIR / "monthly_means_apr_sep_2022_2025.csv")
mm_lookup = {(int(r.year), int(r.month)): float(r.monthly_mean)
             for _, r in mm_df.iterrows()}
print("Monthly means (all years):")
print(mm_df.to_string(index=False))

Monthly means (all years):
 year  month  monthly_mean
 2022      4    235.457251
 2022      5    242.871489
 2022      6    246.627216
 2022      7    250.650039
 2022      8    257.382165
 2022      9    263.555837
 2023      4    263.514095
 2023      5    247.941418
 2023      6    245.098825
 2023      7    249.713148
 2023      8    252.389686
 2023      9    249.455656
 2024      4    243.909244
 2024      5    243.121841
 2024      6    246.040238
 2024      7    257.972610
 2024      8    262.516649
 2024      9    257.544589
 2025      4    245.587309
 2025      5    241.190311
 2025      6    241.269116
 2025      7    241.441229
 2025      8    245.613360
 2025      9    240.437162


## 3. Compute MaxMin spatial ordering & NNS map

`ord_mm` is computed on **grid coordinates** (`Latitude/Longitude`) from the first time step.  
All time steps share the same grid → `ord_mm` is consistent across the entire season.  
The actual covariance uses **source coordinates** (`Source_Latitude/Longitude`) retrieved in `load_year_for_seasonal` — same logic as `keep_ori=True` in the daily model.

In [4]:
# Load day index to get first valid key
idx_df = pd.read_csv(DATA_DIR / "day_index_apr_sep_2022_2025.csv")
year_dates = idx_df[idx_df["year"] == YEAR].sort_values("date_str")
first_date = year_dates.iloc[0]["date_str"]
first_key  = year_dates.iloc[0]["key_h0"]
print(f"Using first date: {first_date}  key: {first_key}")

# Load first time step
with open(DATA_DIR / f"tco_grid_apr_sep_{YEAR}.pkl", "rb") as f:
    merged_first = pickle.load(f)

df0 = merged_first[first_key]

# Filter spatial domain
mask = (
    (df0["Latitude"]  >= LAT_RANGE[0]) & (df0["Latitude"]  <= LAT_RANGE[1]) &
    (df0["Longitude"] >= LON_RANGE[0]) & (df0["Longitude"] <= LON_RANGE[1])
)
df0_filt = df0[mask].dropna(subset=["Latitude", "Longitude"])

locs0 = df0_filt[["Latitude", "Longitude"]].values.astype(np.float64)
print(f"Spatial locations: {locs0.shape[0]} observations")

# MaxMin ordering
t0 = time.time()
ord_mm = orderings.maxmin_cpp(locs0)          # returns index array, length N
locs_ordered = locs0[ord_mm]
print(f"MaxMin ordering done in {time.time()-t0:.2f}s")

# NNS map on MaxMin-ordered locations
t0 = time.time()
nns_map = orderings.find_nns_l2(locs_ordered, max_nn=MM_COND_NUMBER)
print(f"NNS map done in {time.time()-t0:.2f}s  shape={nns_map.shape}")

del merged_first   # free memory

Using first date: 2024-04-01  key: 2024_04_y24m04day01_hm00:56
Spatial locations: 5733 observations
MaxMin ordering done in 0.00s
NNS map done in 0.15s  shape=(5733, 30)


## 4. Load seasonal data

In [5]:
t0 = time.time()

data_map = load_year_for_seasonal(
    year        = YEAR,
    data_dir    = DATA_DIR,
    mm_lookup   = mm_lookup,
    ord_mm      = ord_mm,
    months      = MONTHS,
    max_days    = MAX_DAYS,
    stride_days = STRIDE_DAYS,
)

print(f"Load time: {time.time()-t0:.1f}s")
print(f"Total time steps in data_map: {len(data_map)}")

first_key = next(iter(data_map))
print(f"First key: {first_key}  shape: {data_map[first_key].shape}")
print(f"Columns: lat, lon, O3_centered, Hours_elapsed, D1..D7")

Loaded 2024: 31 days × up to 8 hours = 248 time steps
Load time: 0.3s
Total time steps in data_map: 248
First key: 2024_07_y24m07day01_hm00:53  shape: torch.Size([5733, 11])
Columns: lat, lon, O3_centered, Hours_elapsed, D1..D7


## 5. Instantiate model & precompute conditioning sets

In [6]:
model = VecchiaAR1Daily(
    smooth    = SMOOTH,
    input_map = data_map,
    nns_map   = nns_map,
    k_space   = K_SPACE,
)

t0 = time.time()
model.precompute_conditioning_sets()
print(f"Precompute time: {time.time()-t0:.1f}s")

🚀 VecchiaAR1Daily  k_space=6  [G1=6, G2=13, G3=20 cond pts] [T=248, obs=1420741] ✅ Done. (G1=5732, G2=39975, G3=1375034)
Precompute time: 10.3s


## 6. Initial parameters

Same starting values as `gpu_vecc_fit_dynamic_grid_031826.ipynb` (July 2024 estimates).

In [ ]:
# Physical init values — seasonal model (July 2024)
# range_time: temporal range in hours; 200h ≈ 8 days is reasonable for monthly data
# advec: initial = 0 (raw tanh argument); effective adv bounded to ADV_MAX = 0.002 deg/hr
init_sigmasq    = 13.059
init_range_lat  = 0.20
init_range_lon  = 0.25
init_range_time = 200.0   # hours
init_nugget     = 0.247

# Reparametrize to log(phi) scale
init_phi2 = 1.0 / init_range_lon
init_phi1 = init_sigmasq * init_phi2
init_phi3 = (init_range_lon / init_range_lat) ** 2
init_phi4 = (init_range_lon / init_range_time) ** 2

initial_vals = [
    math.log(init_phi1), math.log(init_phi2), math.log(init_phi3),
    math.log(init_phi4), 0.0, 0.0, math.log(init_nugget)
]

print("Initial params (physical):")
print(_to_physical(initial_vals))

## 7. Quick likelihood check before fitting

In [17]:
with torch.no_grad():
    p = torch.stack(params_list)
    nll0 = model.vecchia_likelihood(p)
print(f"Initial NLL/obs = {nll0.item():.6f}")

Initial NLL/obs = -0.542775


In [18]:
# ── Gradient diagnostic — run before fit ─────────────────────────────────────
import importlib
import GEMS_TCO.kernels_vecchia_seasonal as _kmod
importlib.reload(_kmod)
from GEMS_TCO.kernels_vecchia_seasonal import VecchiaAR1Daily, _to_physical

print("=== Diagnostic ===")
p_stack = torch.stack(params_list)
print(f"params shape      : {p_stack.shape}")
print(f"params req_grad   : {p_stack.requires_grad}")

with torch.enable_grad():
    loss_diag = model.vecchia_likelihood(p_stack)

print(f"loss value        : {loss_diag.item():.6f}")
print(f"loss requires_grad: {loss_diag.requires_grad}")
print(f"loss grad_fn      : {loss_diag.grad_fn}")

# Check each group
for name, X_b in [("G1", model.G1_X), ("G2", model.G2_X), ("G3", model.G3_X)]:
    if X_b is None:
        print(f"{name}: None")
    else:
        print(f"{name}: shape={X_b.shape}  dtype={X_b.dtype}")

=== Diagnostic ===
params shape      : torch.Size([7, 1])
params req_grad   : True
loss value        : -0.542775
loss requires_grad: True
loss grad_fn      : <DivBackward0 object at 0x3dce0d4b0>
G1: shape=torch.Size([5732, 7, 3])  dtype=torch.float64
G2: shape=torch.Size([39975, 14, 3])  dtype=torch.float64
G3: shape=torch.Size([1375034, 21, 3])  dtype=torch.float64


## 8. Fit with L-BFGS

In [ ]:
LBFGS_MAX_STEPS = 30
LBFGS_LR        = 1.0
LBFGS_MAX_ITER  = 20    # inner L-BFGS iterations per step

t0 = time.time()

# fit() now takes initial_vals directly and always creates fresh params internally.
# No need to manage params_list externally — repeated calls are safe.
raw_result, final_loss = model.fit(
    initial_vals   = initial_vals,
    max_steps      = LBFGS_MAX_STEPS,
    lr             = LBFGS_LR,
    tolerance_grad = 1e-7,
    max_iter       = LBFGS_MAX_ITER,
)

elapsed = time.time() - t0
print(f"
Total fit time: {elapsed:.1f}s ({elapsed/60:.1f}min)")

## 9. Results

In [20]:
phys = _to_physical(raw_result)

print(f"\n{'='*55}")
print(f"  Seasonal VecchiaAR1Daily  — Year {YEAR}")
print(f"  stride_days={STRIDE_DAYS}  max_days={MAX_DAYS}")
print(f"  n_obs={model.n_obs:,}  k_space={K_SPACE}  ν={SMOOTH}")
print(f"{'='*55}")
for k, v in phys.items():
    print(f"  {k:<14}: {v}")
print(f"  final NLL/obs : {final_loss:.6f}")
print(f"{'='*55}")


  Seasonal VecchiaAR1Daily  — Year 2024
  stride_days=1  max_days=None
  n_obs=1,420,741  k_space=6  ν=0.5
  sigma_sq      : 1.2489749521320178e+199
  range_lon     : 2.11165062020876e+38
  range_lat     : 1.0488252988518098e+32
  range_time    : 5.0884935035860056e+35
  advec_lat     : 166.7532
  advec_lon     : -233.5882
  nugget        : 0.0
  final NLL/obs : -0.542775


## 10. Save results

In [ ]:
import json

result = {
    "year"        : YEAR,
    "stride_days" : STRIDE_DAYS,
    "max_days"    : MAX_DAYS,
    "n_time_steps": len(data_map),
    "n_obs"       : model.n_obs,
    "k_space"     : K_SPACE,
    "smooth"      : SMOOTH,
    "final_nll"   : final_loss,
    "raw_params"  : raw_result,
    **phys,
}

tag = f"{YEAR}_stride{STRIDE_DAYS}" if MAX_DAYS is None else f"{YEAR}_stride{STRIDE_DAYS}_max{MAX_DAYS}"
out_json = OUT_DIR / f"seasonal_vecc_ar1day_{tag}.json"

with open(out_json, "w") as f:
    json.dump(result, f, indent=2)

print(f"Saved: {out_json}")
print(json.dumps(result, indent=2))